# Transformers

### Connect to datasets (Google-Drive)

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
import os
DATA_DIR = "/work/users/das214/570AI_Tut/smartpix_data/16x16x20"
# DATA_DIR="/content/drive/MyDrive/PHYS570AI_Fall25/smartpix_data/"
os.listdir(DATA_DIR)

['labels_subset_full.csv', 'recon_subset_full.csv']

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

# recon_pos_df = pd.read_csv(f"{DATA_DIR}/recon_positive.csv", header=0)
recon_neg_df = pd.read_csv(f"{DATA_DIR}/recon_subset_full.csv", header=0)
# labels_pos_df = pd.read_csv(f"{DATA_DIR}/labels_positive.csv", header=0)
labels_neg_df = pd.read_csv(f"{DATA_DIR}/labels_subset_full.csv", header=0)

# recon_pos_df = pd.read_csv(f"{DATA_DIR}/recon_positive_large.csv", header=0)
# recon_neg_df = pd.read_csv(f"{DATA_DIR}/recon_negative_large.csv", header=0)
# labels_pos_df = pd.read_csv(f"{DATA_DIR}/labels_positive_large.csv", header=0)
# labels_neg_df = pd.read_csv(f"{DATA_DIR}/labels_negative_large.csv", header=0)

# print(recon_pos_df.shape, labels_pos_df.shape)
print(recon_neg_df.shape, labels_neg_df.shape)

(60000, 5120) (60000, 13)


In [4]:
N_T, N_Y, N_X = 20, 16, 16 # Those are snaps with evolving time
# recon_pos_np = recon_pos_df.to_numpy().reshape(-1, N_T, N_Y, N_X)
recon_neg_np = recon_neg_df.to_numpy().reshape(-1, N_T, N_Y, N_X)

# print(recon_pos_np.shape)
print(recon_neg_np.shape)

(60000, 20, 16, 16)


## Inspecting the dataset

In [5]:
# CHANGE: Align ticks to pixel centers; draw grid at half-integers (cell boundaries)
# REASON: Show labels 0..15 at cell centers while keeping grid lines between pixels.

from ipywidgets import interact, IntSlider
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

def time_slice_slider_view(cube, charge_label="positive"):
    vlim = np.max(np.abs(cube))
    n_t, n_y, n_x = cube.shape

    def show_slice(t):
        fig, ax = plt.subplots(figsize=(6, 5))
        im = ax.imshow(
            cube[t],
            origin="lower",
            aspect="equal",
            cmap="RdBu_r",
            vmin=-vlim,
            vmax=+vlim,
            extent=[-0.5, n_x - 0.5, -0.5, n_y - 0.5]  # pixel centers at integers
        )

        ax.set_title(rf"{charge_label} t = {t} ($\Delta t = 200\,\mathrm{{ps}}$)")
        ax.set_xlabel("x (pixel)")
        ax.set_ylabel("y (pixel)")

        # Major ticks at integer centers (0..n_x-1, 0..n_y-1)
        ax.set_xticks(np.arange(0, n_x, 1))
        ax.set_yticks(np.arange(0, n_y, 1))
        ax.xaxis.set_major_formatter(FormatStrFormatter('%d'))
        ax.yaxis.set_major_formatter(FormatStrFormatter('%d'))

        # Minor ticks at half-integers (cell boundaries), used only for grid
        ax.set_xticks(np.arange(-0.5, n_x, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, n_y, 1), minor=True)
        ax.grid(which='minor', linestyle='--', linewidth=0.5, alpha=0.6)
        ax.tick_params(which='minor', length=0)  # hide minor tick marks

        fig.colorbar(im, ax=ax, label="Charge (e-)")
        fig.tight_layout()
        plt.show()

    interact(show_slice, t=IntSlider(0, 0, n_t - 1, 1))


In [6]:
# For negative sample
data_idx = 25000
time_slice_slider_view(recon_neg_np[data_idx], charge_label="")

interactive(children=(IntSlider(value=0, description='t', max=19), Output()), _dom_classes=('widget-interact',…

In [7]:
from matplotlib.ticker import FormatStrFormatter
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter

def save_cube_animation(
    cube,
    out_path="time_slices.gif",  # .gif → PillowWriter, .mp4 → FFMpegWriter
    fps=4,
    charge_label="negative",
    t_step_ps=200,
    cmap="RdBu_r",
    vlim=None
):
    n_t, n_y, n_x = cube.shape
    if vlim is None:
        vlim = float(np.max(np.abs(cube))) or 1.0

    fig, ax = plt.subplots(figsize=(5.5, 4.8), dpi=150)
    im = ax.imshow(
        cube[0], origin="lower", aspect="equal", cmap=cmap,
        vmin=-vlim, vmax=+vlim,
        extent=[-0.5, n_x - 0.5, -0.5, n_y - 0.5]  # pixel centers at integers
    )
    fig.colorbar(im, ax=ax, label="Charge (a.u.)")
    ax.set_xlabel("x (pixel)"); ax.set_ylabel("y (pixel)")
    ax.set_xticks(np.arange(0, n_x, 1)); ax.set_yticks(np.arange(0, n_y, 1))
    ax.xaxis.set_major_formatter(FormatStrFormatter('%d'))
    ax.yaxis.set_major_formatter(FormatStrFormatter('%d'))
    # grid on half-integers (cell boundaries), not on the labels
    ax.set_xticks(np.arange(-0.5, n_x, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_y, 1), minor=True)
    ax.grid(which='minor', linestyle='--', linewidth=0.5, alpha=0.6)
    ax.tick_params(which='minor', length=0)
    title = ax.set_title(rf"t = 0 ($\Delta t = {t_step_ps}\,\mathrm{{ps}}$)")
    fig.tight_layout()

    def update(t):
        im.set_data(cube[t])
        title.set_text(rf"t = {t} ($\Delta t = {t_step_ps}\,\mathrm{{ps}}$)")
        return (im, title)

    anim = FuncAnimation(fig, update, frames=n_t, interval=1000/fps, blit=False)

    ext = out_path.split('.')[-1].lower()
    if ext == "gif":
        # Requires Pillow (PIL). Usually preinstalled; if not, pip install pillow.
        writer = PillowWriter(fps=fps)
    elif ext in ("mp4", "m4v"):
        # Requires ffmpeg in PATH.
        writer = FFMpegWriter(fps=fps, codec="libx264", bitrate=1800)
    else:
        raise ValueError("out_path must end with .gif or .mp4")

    anim.save(out_path, writer=writer)
    plt.close(fig)
    print(f"Saved {out_path} ({n_t} frames @ {fps} fps)")


In [8]:
os.makedirs('GIF_new/Noise', exist_ok=True)

In [9]:
from tqdm import tqdm
data_indices = [1, 2, 3, 11, 50, 200, 1290, 6515, 25000]

for data_idx in tqdm(data_indices):
    save_cube_animation(recon_neg_np[data_idx], f"GIF_new/time_slices{data_idx}.gif", fps=4, charge_label="negative", t_step_ps=200)


 11%|█         | 1/9 [00:05<00:42,  5.29s/it]

Saved GIF_new/time_slices1.gif (20 frames @ 4 fps)


 22%|██▏       | 2/9 [00:10<00:37,  5.39s/it]

Saved GIF_new/time_slices2.gif (20 frames @ 4 fps)


 33%|███▎      | 3/9 [00:16<00:32,  5.36s/it]

...

Saved GIF_new/time_slices1290.gif (20 frames @ 4 fps)


 89%|████████▉ | 8/9 [00:42<00:05,  5.34s/it]

Saved GIF_new/time_slices6515.gif (20 frames @ 4 fps)


100%|██████████| 9/9 [00:48<00:00,  5.34s/it]

Saved GIF_new/time_slices25000.gif (20 frames @ 4 fps)


In [10]:
data_idx = 1

noise_std = 80.0
noise = np.random.normal(loc=0.0, scale=noise_std, size=recon_neg_np[data_idx].shape)

time_slice_slider_view(recon_neg_np[data_idx] + noise, charge_label="")

interactive(children=(IntSlider(value=0, description='t', max=19), Output()), _dom_classes=('widget-interact',…

In [11]:
for data_idx in tqdm(data_indices):
    noise = np.random.normal(loc=0.0, scale=noise_std, size=recon_neg_np[data_idx].shape)
    save_cube_animation(recon_neg_np[data_idx] + noise, f"GIF_new/Noise/time_slices{data_idx}.gif", fps=4, charge_label="negative", t_step_ps=200)


 11%|█         | 1/9 [00:05<00:43,  5.45s/it]

Saved GIF_new/Noise/time_slices1.gif (20 frames @ 4 fps)


 22%|██▏       | 2/9 [00:10<00:37,  5.35s/it]

Saved GIF_new/Noise/time_slices2.gif (20 frames @ 4 fps)


 33%|███▎      | 3/9 [00:16<00:32,  5.38s/it]

...

Saved GIF_new/Noise/time_slices1290.gif (20 frames @ 4 fps)


 89%|████████▉ | 8/9 [00:43<00:05,  5.37s/it]

Saved GIF_new/Noise/time_slices6515.gif (20 frames @ 4 fps)


100%|██████████| 9/9 [00:48<00:00,  5.37s/it]

Saved GIF_new/Noise/time_slices25000.gif (20 frames @ 4 fps)
